# 02 · Datos históricos de inflación y tasas

Prepara referencias históricas para contrastar los supuestos ilustrativos del [modelo de obligaciones](01_liability_model.ipynb). Ejecuta las celdas en orden con Python 3 y conexión a internet para la primera descarga. Se usa la biblioteca estándar de Python.

| Serie mensual | Fuente primaria | Valor original | Transformación |
| --- | --- | --- | --- |
| [CPIAUCNS](https://fred.stlouisfed.org/series/CPIAUCNS) | Bureau of Labor Statistics, vía FRED | IPC general de EE. UU., sin ajuste estacional, 1982–1984 = 100 | Inflación interanual: IPC del mes / IPC del mismo mes del año anterior − 1 |
| [GS10](https://fred.stlouisfed.org/series/GS10) | Federal Reserve Board, vía FRED | Rendimiento nominal del Tesoro a 10 años, promedio mensual, en porcentaje | Dividir entre 100 para expresar en decimal |

La ventana de análisis es enero de 1990 a diciembre de 2024. Se descarga el IPC desde enero de 1989 para medir la inflación interanual desde el primer mes. La ventana fija permite comparar ejecuciones. El hash del archivo distingue descargas con revisiones históricas; conserva el CSV local si necesitas reproducir exactamente los mismos datos.

In [ ]:
from csv import DictReader
from datetime import date
from hashlib import sha256
from io import StringIO
from math import isfinite
from pathlib import Path
from statistics import median
from urllib.parse import urlencode
from urllib.request import urlopen

START = date(1990, 1, 1)
END = date(2024, 12, 1)
CPI_START = START.replace(year=START.year - 1)
CACHE = Path("fred_snapshots")


def month_range(start, end):
    current = start
    while current <= end:
        yield current
        if current.month == 12:
            current = date(current.year + 1, 1, 1)
        else:
            current = date(current.year, current.month + 1, 1)


def parse_fred(raw, series, start, end):
    reader = DictReader(StringIO(raw.decode("utf-8-sig")))
    fields = reader.fieldnames or []
    date_column = next((key for key in ("observation_date", "DATE") if key in fields), None)
    if date_column is None or series not in fields:
        raise ValueError(f"{series}: encabezado CSV inesperado: {fields}")

    values = {}
    for row in reader:
        month = date.fromisoformat(row[date_column])
        if not start <= month <= end:
            continue
        if month.day != 1 or month in values:
            raise ValueError(f"{series}: fecha mensual inválida o duplicada: {month}")
        value = row[series].strip()
        values[month] = None if value in ("", ".") else float(value)
        if values[month] is not None and not isfinite(values[month]):
            raise ValueError(f"{series}: valor no finito en {month}")

    missing = [month for month in month_range(start, end) if values.get(month) is None]
    if missing:
        example = ", ".join(str(month) for month in missing[:5])
        raise ValueError(f"{series}: faltan {len(missing)} meses, por ejemplo {example}")
    return values


def load_fred(series, start, end):
    params = urlencode({"id": series, "cosd": start.isoformat(), "coed": end.isoformat()})
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?{params}"
    path = CACHE / f"{series}_{start}_{end}.csv"
    from_cache = path.is_file()
    if from_cache:
        raw = path.read_bytes()
    else:
        with urlopen(url, timeout=30) as response:
            raw = response.read()

    values = parse_fred(raw, series, start, end)
    if not from_cache:
        CACHE.mkdir(parents=True, exist_ok=True)
        path.write_bytes(raw)

    provenance = {
        "series": series,
        "url": url,
        "file": str(path),
        "sha256": sha256(raw).hexdigest(),
        "origin": "archivo local" if from_cache else "descarga FRED",
    }
    return values, provenance


def historical_panel(cpi, treasury, months):
    if any(value <= 0 for value in cpi.values()):
        raise ValueError("El nivel del IPC debe ser positivo.")
    return [
        (month, cpi[month] / cpi[month.replace(year=month.year - 1)] - 1,
         treasury[month] / 100)
        for month in months
    ]

In [ ]:
from math import isclose

sample = b"observation_date,GS10\n2024-01-01,4.2\n2024-02-01,4.5\n"
first, last = date(2024, 1, 1), date(2024, 2, 1)
parsed = parse_fred(sample, "GS10", first, last)
assert parsed == {first: 4.2, last: 4.5}
assert len(list(month_range(START, END))) == 420
try:
    parse_fred(sample.replace(b"4.5", b"."), "GS10", first, last)
except ValueError as error:
    assert "faltan 1 meses" in str(error)
else:
    raise AssertionError("El parser aceptó un mes sin dato.")

_, inflation, yield_10y = historical_panel(
    {date(2023, 1, 1): 100, first: 103}, parsed, [first]
)[0]
assert isclose(inflation, 0.03) and isclose(yield_10y, 0.042)

In [ ]:
cpi, cpi_source = load_fred("CPIAUCNS", CPI_START, END)
treasury, treasury_source = load_fred("GS10", START, END)
panel = historical_panel(cpi, treasury, month_range(START, END))

for source in (cpi_source, treasury_source):
    print(source)

print(f"Meses con ambas métricas: {len(panel)}")
for index, label in ((1, "Inflación interanual"), (2, "Rendimiento nominal a 10 años")):
    values = [row[index] for row in panel]
    print(f"{label}: mínimo {min(values):.2%}, mediana {median(values):.2%}, "
          f"máximo {max(values):.2%}")

## Uso y limitaciones

Los resúmenes se generan al ejecutar la descarga y describen la ventana histórica seleccionada. Los meses faltantes bloquean el cálculo para evitar interpolaciones implícitas. Las observaciones de inflación interanual de meses consecutivos comparten once meses de información.

GS10 aporta contexto histórico sobre rendimientos nominales. Elegir una tasa o curva de descuento para los 360 retiros requiere considerar los plazos de los flujos y justificar la convención. El rendimiento a diez años tampoco representa el retorno total de un portafolio.

**Próximo paso.** Definir universo, frecuencia y fuentes de retornos totales para [03_return_analysis.ipynb](03_return_analysis.ipynb). Para un backtest, documentar qué datos estaban disponibles en cada fecha.